In [6]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import numpy as np

### Data preprocessing

In [7]:
digits = load_digits()
X = digits.data
y = digits.target

In [8]:
X[0]

array([ 0.,  0.,  5., 13.,  9.,  1.,  0.,  0.,  0.,  0., 13., 15., 10.,
       15.,  5.,  0.,  0.,  3., 15.,  2.,  0., 11.,  8.,  0.,  0.,  4.,
       12.,  0.,  0.,  8.,  8.,  0.,  0.,  5.,  8.,  0.,  0.,  9.,  8.,
        0.,  0.,  4., 11.,  0.,  1., 12.,  7.,  0.,  0.,  2., 14.,  5.,
       10., 12.,  0.,  0.,  0.,  0.,  6., 13., 10.,  0.,  0.,  0.])

In [9]:
X = X / 16.0  # Normalize pixel values to [0, 1]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
# set with target is one hot encoded
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y.reshape(-1, 1))
X_train_oh, X_test_oh, y_train_oh, y_test_oh = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42
)

### DENSE LAYER CLASS (w/o inbuilt activations)

In [ ]:
class Layer_Dense:
    # Layer initialization
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    # Forward pass
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(self.weights, inputs) + self.biases
        

### Activation functions

#### 01- Relu

In [ ]:
# Relu activation function
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
    
    def backward(self, dvalues):
        # Derivative of ReLU
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

#### 02- Sigmoid

In [ ]:
# Sigmoid activation function
class Activation_Sigmoid:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = 1 / (1 + np.exp(-inputs))
    
    def backward(self, dvalues):
        # Derivative of sigmoid
        self.dinputs = dvalues * (1 - self.output) * self.output

#### 03- Softmax

In [ ]:
# Softmax activation function
class Activation_Softmax:
    def forward(self, inputs):
        self.inputs = inputs
        # Subtract max for numerical stability
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)
    
    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)
        for i, (single_output, single_dvalue) in enumerate(zip(self.output, dvalues)):
            # Flatten output array
            single_output = single_output.reshape(-1, 1)
            # Jacobian matrix of the softmax function
            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)
            self.dinputs[i] = np.dot(jacobian_matrix, single_dvalue)

#### 04- Tanh

In [21]:
# Tanh activation function
class Activation_Tanh:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.tanh(inputs)

#### 05- Leaky_Relu

In [22]:
# Leaky ReLU activation function
class Activation_LeakyReLU:
    def __init__(self, alpha=0.01):
        self.alpha = alpha

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.where(inputs > 0, inputs, self.alpha * inputs)

### Loss Functions

#### 01- Logloss

In [ ]:
# Common loss class
class Loss:
 def calculate(self, output, y):
  # Calculate sample losses
  sample_losses = self.forward(output, y)
  # Calculate mean loss
  data_loss = np.mean(sample_losses)
  # Return loss
  return data_loss

#### 02- CategoricalCrossEntropy

In [ ]:
# Categorical Cross-Entropy Loss
class Loss_CategoricalCrossEntropy:
    def forward(self, y_pred, y_true):
        # Avoid log(0)
        y_pred_clipped = np.clip(y_pred, 1e-12, 1 - 1e-12)

        # If labels are one-hot encoded
        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)
        else:
            correct_confidences = y_pred_clipped[np.arange(len(y_pred)), y_true]

        losses = -np.log(correct_confidences)
        return np.mean(losses)

    def backward(self, y_pred, y_true):
        samples = y_pred.shape[0]
        labels = y_pred.shape[1]

        # If labels are one-hot encoded, convert to sparse
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)

        # Gradient: subtract 1 from the correct class
        dinputs = y_pred.copy()
        dinputs[np.arange(samples), y_true] -= 1
        dinputs /= samples
        return dinputs


#### 03- Mean Squared Error (MSE)

In [35]:
# Mean Squared Error Loss
class Loss_MeanSquaredError:
    def forward(self, y_pred, y_true):
        # Store input values
        self.y_pred = y_pred
        self.y_true = y_true

        # Compute MSE
        loss = np.mean((y_true - y_pred) ** 2)
        return loss

    def backward(self, y_pred, y_true):
        # Number of samples
        samples = y_true.shape[0]

        # Gradient of MSE
        dinputs = -2 * (y_true - y_pred) / samples
        return dinputs


#### 04- Mean Absolute Error (MAE)

In [34]:
# Mean Absolute Error Loss
class Loss_MeanAbsoluteError:
    def forward(self, y_pred, y_true):
        self.y_pred = y_pred
        self.y_true = y_true

        # Compute MAE
        loss = np.mean(np.abs(y_true - y_pred))
        return loss

    def backward(self, y_pred, y_true):
        samples = y_true.shape[0]

        # Gradient of MAE
        dinputs = np.sign(y_pred - y_true) / samples
        return dinputs


### Accuracy calculation

In [ ]:
def accuracy_classification(y_pred, y_true):
    # If predictions are probabilities (softmax), get the class with highest score
    y_pred_labels = np.argmax(y_pred, axis=1)

    # If y_true is one-hot, convert to class indices
    if len(y_true.shape) == 2:
        y_true = np.argmax(y_true, axis=1)

    accuracy = np.mean(y_pred_labels == y_true)
    return accuracy